# Validation Test - Oscillating Droplet

This notebook and the corresponding evaluation notebook (DropletOscillating_Evaluation.ipynb) are part of published results (cf. 7.2.2) found in *On a marching level-set method for extended discontinuous Galerkin methods for incompressible two-phase flows: Application to two-dimensional settings* (https://doi.org/10.1002%2Fnme.6853).

In [ ]:
#r "BoSSSpad.dll"
//#r "C:\BoSSS-experimental\public\src\L4-application\BoSSSpad\bin\Release\net8.0\BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
ExecutionQueues

index,type,value
0,BoSSS.Application.BoSSSpad.MiniBatchProcessorClient,MiniBatchProcessor client LocalPC @C:\BoSSS-localJobs\binaries
1,BoSSS.Application.BoSSSpad.MsHPC2012Client,"MS HPC client FDY-WindowsHPC @DC3, @\\dc3\userspace\smuda\hpccluster\binaries"


In [ ]:
string userName = System.Security.Principal.WindowsIdentity.GetCurrent().Name;
userName

FDY\smuda

In [ ]:
//var myBatch = ExecutionQueues[1];
var myBatch = (userName.Equals(@"FDY\jenkinsci")) ? GetDefaultQueue() : ExecutionQueues[1];
myBatch

RuntimeLocation,win\amd64
AdditionalEnvironmentVars,[ ]
DeploymentBaseDirectory,\\dc3\userspace\smuda\hpccluster\binaries
DeployRuntime,True
Name,FDY-WindowsHPC
DotnetRuntime,dotnet
Username,FDY\smuda
NumOfServiceCoresPerMPIprocess,1
ServerName,DC3
ComputeNodes,<null>
DefaultJobPriority,Normal


In [ ]:
wmg.Init("XNSEpaper_Droplet", myBatch);

Project name is set to 'XNSEpaper_DropletInEquilibrium'.
Opening existing database '\\dc3\userspace\smuda\hpccluster\XNSEpaper_DropletInEquilibrium'.


In [ ]:
wmg.SetNameBasedSessionJobControlCorrelation();

In [5]:
using BoSSS.Solution.NSECommon;
using BoSSS.Solution.XNSECommon;
using BoSSS.Solution.Timestepping;
using BoSSS.Solution.XdgTimestepping;
using BoSSS.Solution.LevelSetTools;
using BoSSS.Solution.LevelSetTools.SolverWithLevelSetUpdater;
using BoSSS.Application.XNSE_Solver.Logging;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

## Sessions

In [6]:
var sessions = BoSSSshell.WorkflowMgm.Sessions;
sessions

#0: DropletOscillating	DropletOscillating_La500_meshStudy_k2_mesh10	7/30/2025 11:17:05 AM	41095fc3...
#1: DropletOscillating	DropletOscillating_La500_meshStudy_k2_mesh80*	7/30/2025 11:17:47 AM	b44ec82d...
#2: DropletOscillating	DropletOscillating_La500_meshStudy_k2_mesh60*	7/30/2025 11:17:34 AM	3555b4ee...
#3: DropletOscillating	DropletOscillating_La500_meshStudy_k2_mesh40*	7/30/2025 11:17:25 AM	d00ecc37...
#4: DropletOscillating	DropletOscillating_La500_meshStudy_k2_mesh20*	7/30/2025 11:17:15 AM	ae17b0ae...


In [ ]:
// foreach (var s in sessions) {
//     if(s.Name.Contains("DropletOscillation") && s.CreationTime.Date == new DateTime(2025, 11, 10)) {
//         s.Delete(true);
//         //Console.WriteLine($"Deleted session from {s.CreationTime.Date}");
//     }
// }

In [ ]:
// var sessions = BoSSSshell.WorkflowMgm.Sessions;
// sessions

## Grid Creation for Convergence Study

In [7]:
int[] Resolutions = new int[] { 10, 20, 40, 60, 80 };
IGridInfo[] Grids = new IGridInfo[Resolutions.Length];

double L = 0.5; 

for(int i = 0; i < Resolutions.Length; i++) {
    int Res = Resolutions[i];
    string GridName = $"DropletOscillating_{Res}";

    IGridInfo cachedGrid = wmg.Grids.FirstOrDefault(grid => grid.Name == GridName);
    cachedGrid = null;
    if(cachedGrid == null) {
        
        // must create new Grid
        double[] xNodes = GenericBlas.Linspace(-L, L, Res + 1);
        double[] yNodes = xNodes;
        var grd = Grid2D.Cartesian2DGrid(xNodes, yNodes);
    
        grd.Name = GridName;
    
        grd.DefineEdgeTags(delegate (double[] X) {
            string ret = null;
            if (Math.Abs(X[0] + L) <= 1.0e-8)
                ret = IncompressibleBcType.Wall.ToString();
            if (Math.Abs(X[0] - L) <= 1.0e-8)
                ret = IncompressibleBcType.Wall.ToString();
            if (Math.Abs(X[1] + L) <= 1.0e-8)
                ret = IncompressibleBcType.Wall.ToString();
            if (Math.Abs(X[1] - L) <= 1.0e-8)
                ret = IncompressibleBcType.Wall.ToString();
            return ret;
        });
        
        Grids[i] = wmg.SaveGrid(grd);
        
    } else {
        //Console.WriteLine($"type: {cachedGrid.GetType()}, is IGridInfo? {cachedGrid is IGridInfo}");
        Console.WriteLine("Grid already found in database - identifid by name " + GridName);
        Grids[i] = cachedGrid;
    }
    
}

## Initial Values

In [1]:
double r = 0.25;

In [8]:
double a = 1.25;
a

1.25

In [9]:
double b = 0.8;
b

0.8

In [ ]:
var PhiFunc = new Formula(
    "Phi",
    false,
    "using ilPSP.Utils; " + 
    "double r = 0.25;" + 
    "double a = 1.25*r;" + 
    "double b = 0.8*r;" + 
    "double Phi(double[] X) { " + 
    "    return Math.Sqrt((X[0] / a).Pow2() + (X[1] / b).Pow2()) - 1.0; " + 
    "}");

## Physical Settings

In [11]:
double rho = 1e4;    
double mu = 1.0;
double sigma = 0.1;

double r = 0.25; // equilibrium drop radius
double La = sigma * rho * 2 * r / mu.Pow2();
La

500

In [12]:
double dt = 0.5;
double t_end = 1000;

## Control settings

In [13]:
int[] degS = new int[] { 2 };

In [14]:
List<XNSE_Control> Controls = new List<XNSE_Control>();

In [ ]:
for(int iDeg = 0; iDeg < degS.Length; iDeg++) {
for(int iGrd = 0; iGrd < Grids.Length; iGrd++) {
    
    XNSE_Control C = new XNSE_Control();
    
    int pDeg = degS[iDeg];   
    var grd  = Grids[iGrd];

    C.SetDGdegree(pDeg);
    
    // set grid
    C.SetGrid(grd);
    
    // initial conditions   
    C.AddInitialValue("Phi", PhiFunc);   
    
    C.PostprocessingModules.Add(new Dropletlike(){ SolverStage = 2, LogPeriod = 4});
    C.PostprocessingModules.Add(new EnergyLogging(){ SolverStage = 2, LogPeriod = 4});

    C.PhysicalParameters.rho_A = rho;
    C.PhysicalParameters.rho_B = rho;
    C.PhysicalParameters.mu_A  = mu;
    C.PhysicalParameters.mu_B  = mu;
    C.PhysicalParameters.Sigma = sigma;

    C.PhysicalParameters.IncludeConvection = true;
    C.PhysicalParameters.Material = true;

    C.CutCellQuadratureType = CutCellQuadratureMethod.Saye;

    C.AdvancedDiscretizationOptions.SST_isotropicMode = SurfaceStressTensor_IsotropicMode.LaplaceBeltrami_ContactLine;
    C.LSContiProjectionMethod = ContinuityProjectionOption.ConstrainedDG;

    C.NonLinearSolver.MaxSolverIterations = 50;
    
    C.TimesteppingMode = AppControl._TimesteppingMode.Transient;
    C.Timestepper_LevelSetHandling = LevelSetHandling.Coupled_Once;
    C.Option_LevelSetEvolution = LevelSetEvolution.FastMarching;
    C.TimeSteppingScheme = TimeSteppingScheme.BDF3;
  
    C.dtMin         = dt;
    C.dtMax         = dt;
    C.Endtime       = t_end;
    C.NoOfTimesteps = (int)(t_end/dt); 
    
    C.saveperiod = 20;
    
    C.SessionName = "DropletOscillation_meshStudy_mesh" + Resolutions[iGrd];
    
    Controls.Add(C);
}
}

In [ ]:
int NoCtrls = Controls.Count();
NoCtrls

[ DropletOscillating_La500_meshStudy_k2_mesh10, DropletOscillating_La500_meshStudy_k2_mesh20, DropletOscillating_La500_meshStudy_k2_mesh40, DropletOscillating_La500_meshStudy_k2_mesh60, DropletOscillating_La500_meshStudy_k2_mesh80 ]

## Launch Jobs

In [ ]:
foreach(var ctrl in Controls) {
    var oneJob              = ctrl.CreateJob();
    
    oneJob.NumberOfMPIProcs = 1;

    //System.Environment.SetEnvironmentVariable("OMP_NUM_THREADS", "2");
    int numThreads = 1;
    oneJob.NumberOfThreads = numThreads;
    IDictionary<string, string> args = oneJob.EnvironmentVars;
    args["BOSSS_ARG_7"] = numThreads.ToString();
    args.Remove("OMP_NUM_THREADS");
    oneJob.MySetCommandLineArguments(args.Values.ToArray());

    //oneJob.UseComputeNodesExclusive = true;\n",

    //((SlurmClient)myBatch).ExecutionTime = "24:00:00";

    oneJob.RetryCount = 1;
    oneJob.Activate(myBatch);
}

Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job DropletOscillating_La500_meshStudy_k2_mesh10 ... 
Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\smuda\hpccluster\binaries\DropletOscillating-XNSE_Solver2025Jul30_111648.441469
copied 51 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.
Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job DropletOscillating_La500_meshStudy_k2_mesh20 ... 
Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\smuda\hpccluster\binaries\DropletOscillating-XNSE_Solver2025Jul30_111658.881900
copied 51 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.
Deployments so far (0): ;
Success: 0
job submit count: 0


...

Deploying job DropletOscillating_La500_meshStudy_k2_mesh40 ... 
Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\smuda\hpccluster\binaries\DropletOscillating-XNSE_Solver2025Jul30_111708.895024
copied 51 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.
Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job DropletOscillating_La500_meshStudy_k2_mesh60 ... 
Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\smuda\hpccluster\binaries\DropletOscillating-XNSE_Solver2025Jul30_111718.812486
copied 51 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.
Deployments so far (0): ;
Success: 0
job submit count: 0


unable to determine job status - unknown


Deploying job DropletOscillating_La500_meshStudy_k2_mesh80 ... 
Deploying executables and additional files ...
Deployment directory: \\dc3\userspace\smuda\hpccluster\binaries\DropletOscillating-XNSE_Solver2025Jul30_111730.829608
copied 51 files.
   written file: control.obj
   copied 'win\amd64' runtime.
deployment finished.


## Wait for Completion and Check Job Status

In [ ]:
int NoInProgress = Controls.Select(ctrl => ctrl.GetJob()).Where(job => job.Status == JobStatus.InProgress 
                                                                || job.Status == JobStatus.PendingInExecutionQueue
                                                                || job.Status == JobStatus.PreActivation).Count();
NoInProgress

In [ ]:
int maxDays = 7;
int waitingDays = 0;
while (NoInProgress > 0 && waitingDays < maxDays) {
    wmg.BlockUntilAllJobsTerminate(3600*24*1); // wait one day for the jobs to finish
    NoInProgress = Controls.Select(ctrl => ctrl.GetJob()).Where(job => job.Status == JobStatus.InProgress).Count();
    waitingDays++;
}

In [ ]:
int NoFailed = Controls.Select(ctrl => ctrl.GetJob()).Where(job => job.Status == JobStatus.FailedOrCanceled).Count();
NoFailed

In [ ]:
int NoSuccess = Controls.Select(ctrl => ctrl.GetJob()).Where(job => job.Status == JobStatus.FinishedSuccessful).Count();
NoSuccess

In [ ]:
// check for restart sessions
if (NoFailed > 0) {
    foreach (var ctrl in Controls) {
        var job = ctrl.GetJob();
        if (job.Status == JobStatus.FailedOrCanceled) {
            var studySess = sessions.Where(sess => sess.Name.Contains(ctrl.SessionName));
            int studyCount = studySess.Count();

            if (studyCount > 1) {
                Console.WriteLine("Restart session found! Take last run");       
                bool success = studySess.Where(sess => sess.Name.EndsWith($"_restart{studyCount-1}")).Single().SuccessfulTermination;
                if (success) {
                    Console.WriteLine($"Restart session {ctrl.SessionName}_restart{studyCount-1} was successful.");
                    NoFailed--;
                    NoSuccess++;
                } else {
                    Console.WriteLine($"Restart session {ctrl.SessionName}_restart{studyCount-1} also failed.");
                }
            } else if (studyCount == 1) {
                Console.WriteLine("No restart session found. {ctrl.SessionName} might to be restarted.");
            } else { // studyCount == 0
                throw new ApplicationException($"No session found for {ctrl.SessionName}!"); // should not happen
            } 
        }
    }

}

In [ ]:
NUnit.Framework.Assert.AreEqual(0, NoFailed, $"Some simulations failed.");

In [ ]:
NUnit.Framework.Assert.AreEqual(NoCtrls, NoSuccess, $"Not all simulations finished successfully.");

Error: (1,37): error CS0103: The name 'NoSuccess' does not exist in the current context